# SOXL Backtest January and June Window Usage

This notebook turns the current production/dashboard backtest data into two reusable 2026 windows:

- `ytd_2026_common`: starts from the first 2026 trading day on or after January 1.
- `june_2026_common`: starts from June 1.

The model comparison uses the dashboard replay `safe_backtest_end`, because qlib daily replays require a following calendar boundary. The market-only readout separately uses the latest canonical market date.

In [1]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    for parent in Path.cwd().parents:
        if (parent / "AGENTS.md").exists():
            ROOT = parent
            break

EXPERIMENT = "soxl_backtest_jan_june_windows_v1"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "runtime/production/card" / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CARD = ROOT / "runtime/production/card/current_model_ytd_june_comparison_v1"
CURRENT_DAILY_PATH = SOURCE_CARD / "current_model_daily.csv"
ALL_MODEL_DAILY_PATH = SOURCE_CARD / "all_model_daily_nav.csv"
FRESHNESS_PATH = ROOT / "dashboard_data/model_replay_freshness.json"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"

print(f"root={ROOT}")
print(f"output_dir={OUTPUT_DIR.relative_to(ROOT)}")

root=/Users/rziiiii/dev/workspace
output_dir=runtime/production/card/soxl_backtest_jan_june_windows_v1


In [2]:
required = [CURRENT_DAILY_PATH, ALL_MODEL_DAILY_PATH, FRESHNESS_PATH, PANEL_PATH]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required source artifacts: {missing}")

freshness = json.loads(FRESHNESS_PATH.read_text())
safe_backtest_end = pd.Timestamp(str(freshness["safe_backtest_end"])).date()
latest_data_date = pd.Timestamp(str(freshness["latest_data_date"])).date()

current_daily = pd.read_csv(CURRENT_DAILY_PATH, parse_dates=["datetime"])
all_model_daily = pd.read_csv(ALL_MODEL_DAILY_PATH, parse_dates=["datetime"])
panel = pd.read_csv(PANEL_PATH, parse_dates=["date"])

print(f"safe_backtest_end={safe_backtest_end}")
print(f"latest_data_date={latest_data_date}")
print(f"current_model_rows={len(current_daily)} all_model_rows={len(all_model_daily)} panel_rows={len(panel)}")

safe_backtest_end=2026-06-09
latest_data_date=2026-06-10
current_model_rows=1617 all_model_rows=22638 panel_rows=4088


In [3]:
def max_drawdown_pct(series: pd.Series) -> float:
    values = series.astype(float)
    drawdown = values / values.cummax() - 1.0
    return float(abs(drawdown.min()) * 100.0)


def model_window(frame: pd.DataFrame, label: str, start: str, end: str) -> pd.DataFrame:
    sub = frame[(frame["datetime"] >= pd.Timestamp(start)) & (frame["datetime"] <= pd.Timestamp(end))].copy()
    if sub.empty:
        raise ValueError(f"No model rows for {label}: {start} to {end}")
    sub.insert(0, "window", label)
    sub["strategy_nav_rebased"] = sub["strategy_nav"] / sub["strategy_nav"].iloc[0] * 100.0
    sub["buyhold_nav_rebased"] = sub["buyhold_nav"] / sub["buyhold_nav"].iloc[0] * 100.0
    sub["strategy_return_pct_from_window_start"] = sub["strategy_nav_rebased"] - 100.0
    sub["soxl_return_pct_from_window_start"] = sub["buyhold_nav_rebased"] - 100.0
    sub["excess_return_pct_from_window_start"] = (
        sub["strategy_return_pct_from_window_start"] - sub["soxl_return_pct_from_window_start"]
    )
    return sub


def model_window_metrics(frame: pd.DataFrame, label: str, start: str, end: str) -> dict:
    sub = model_window(frame, label, start, end)
    strategy = sub["strategy_nav"].astype(float)
    buyhold = sub["buyhold_nav"].astype(float)
    actions = sub["action"].fillna("HOLD").astype(str) if "action" in sub else pd.Series([], dtype=str)
    return {
        "window": label,
        "requested_start": start,
        "start": sub["datetime"].iloc[0].date().isoformat(),
        "end": sub["datetime"].iloc[-1].date().isoformat(),
        "rows": int(len(sub)),
        "soxl_buyhold_return_pct": float((buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0),
        "model_return_pct": float((strategy.iloc[-1] / strategy.iloc[0] - 1.0) * 100.0),
        "excess_return_pct": float((strategy.iloc[-1] / strategy.iloc[0] - buyhold.iloc[-1] / buyhold.iloc[0]) * 100.0),
        "soxl_buyhold_max_drawdown_pct": max_drawdown_pct(buyhold),
        "model_max_drawdown_pct": max_drawdown_pct(strategy),
        "average_position_pct": float(sub["position"].astype(float).mean() * 100.0),
        "end_position_pct": float(sub["position"].astype(float).iloc[-1] * 100.0),
        "position_change_count": int(sub["position"].astype(float).diff().abs().gt(0.001).sum()),
        "non_hold_action_count": int(actions.ne("HOLD").sum()) if len(actions) else 0,
    }


def market_window_metrics(frame: pd.DataFrame, label: str, start: str, end: str) -> dict:
    sub = frame[(frame["date"] >= pd.Timestamp(start)) & (frame["date"] <= pd.Timestamp(end))].copy()
    if sub.empty:
        raise ValueError(f"No market rows for {label}: {start} to {end}")
    price = sub["soxl_close"].astype(float)
    return {
        "window": label,
        "requested_start": start,
        "start": sub["date"].iloc[0].date().isoformat(),
        "end": sub["date"].iloc[-1].date().isoformat(),
        "rows": int(len(sub)),
        "soxl_buyhold_return_pct": float((price.iloc[-1] / price.iloc[0] - 1.0) * 100.0),
        "soxl_buyhold_max_drawdown_pct": max_drawdown_pct(price),
    }

In [4]:
windows = {
    "ytd_2026_common": ("2026-01-01", safe_backtest_end.isoformat()),
    "june_2026_common": ("2026-06-01", safe_backtest_end.isoformat()),
}
market_windows = {
    "ytd_2026_market_latest": ("2026-01-01", latest_data_date.isoformat()),
    "june_2026_market_latest": ("2026-06-01", latest_data_date.isoformat()),
}

window_definitions = pd.DataFrame(
    [
        {"window": label, "requested_start": start, "requested_end": end, "scope": "model_common"}
        for label, (start, end) in windows.items()
    ]
    + [
        {"window": label, "requested_start": start, "requested_end": end, "scope": "market_latest"}
        for label, (start, end) in market_windows.items()
    ]
)
window_definitions

,window,requested_start,requested_end,scope
0,ytd_2026_common,2026-01-01,2026-06-09,model_common
1,june_2026_common,2026-06-01,2026-06-09,model_common
2,ytd_2026_market_latest,2026-01-01,2026-06-10,market_latest
3,june_2026_market_latest,2026-06-01,2026-06-10,market_latest


In [5]:
current_slices = []
current_metrics = []
for label, (start, end) in windows.items():
    current_slices.append(model_window(current_daily, label, start, end))
    current_metrics.append(model_window_metrics(current_daily, label, start, end))

current_window_daily = pd.concat(current_slices, ignore_index=True)
current_metrics_df = pd.DataFrame(current_metrics)
current_metrics_df

,window,requested_start,start,end,rows,soxl_buyhold_return_pct,model_return_pct,excess_return_pct,soxl_buyhold_max_drawdown_pct,model_max_drawdown_pct,average_position_pct,end_position_pct,position_change_count,non_hold_action_count
0,ytd_2026_common,2026-01-01,2026-01-02,2026-06-09,109,326.926303,3.250352,-323.675951,43.473423,21.087602,28.526957,0.0,6,6
1,june_2026_common,2026-06-01,2026-06-01,2026-06-09,7,-11.165928,0.000000,11.165928,34.932634,0.000000,0.000000,0.0,0,0


In [6]:
all_metric_rows = []
for (profile, status, model_kind, source_path), group in all_model_daily.groupby(
    ["profile", "status", "model_kind", "source_path"], dropna=False
):
    group = group.sort_values("datetime")
    for label, (start, end) in windows.items():
        row = model_window_metrics(group, label, start, end)
        row.update(
            {
                "profile": profile,
                "status": status,
                "model_kind": model_kind,
                "source_path": source_path,
            }
        )
        all_metric_rows.append(row)

all_model_metrics_df = pd.DataFrame(all_metric_rows)
all_model_metrics_df = all_model_metrics_df.sort_values(
    ["window", "model_return_pct", "model_max_drawdown_pct"],
    ascending=[True, False, True],
).reset_index(drop=True)

rank_cols = [
    "window",
    "profile",
    "status",
    "model_kind",
    "model_return_pct",
    "soxl_buyhold_return_pct",
    "excess_return_pct",
    "model_max_drawdown_pct",
    "average_position_pct",
    "position_change_count",
]
all_model_metrics_df[rank_cols].head(20)

,window,profile,status,model_kind,model_return_pct,soxl_buyhold_return_pct,excess_return_pct,model_max_drawdown_pct,average_position_pct,position_change_count
0,june_2026_common,v19reg_v22_budget_usage_bull_guard_v1,production_composer,runtime_composer,0.000000,-11.165928,11.165928,0.000000,0.000000,0
1,june_2026_common,balanced_rule_residual_v19,archived,dashboard_replay,-7.281488,-11.165928,3.884440,24.399161,0.000000,0
2,june_2026_common,balanced_rule_residual_v19_reg,candidate,dashboard_replay,-7.312994,-11.165928,3.852934,24.490616,0.000000,0
3,june_2026_common,balanced_rule_residual_v19_reg_light,candidate,dashboard_replay,-7.537298,-11.165928,3.628630,25.138684,0.000000,0
4,june_2026_common,balanced_macro_v1,candidate,dashboard_replay,-7.715885,-11.165928,3.450043,29.901822,0.000000,0
5,june_2026_common,balanced_rule_residual_imitation_v4,candidate,dashboard_replay,-7.838079,-11.165928,3.327849,25.999446,0.000000,0
6,june_2026_common,balanced_rule_residual_v22,canonical,dashboard_replay,-7.936184,-11.165928,3.229744,26.278174,0.000000,0
7,june_2026_common,balanced_rule_residual_v22_continued_retention_v1,ablation,dashboard_replay,-7.936184,-11.165928,3.229744,26.278174,0.000000,0
8,june_2026_common,balanced_rule_residual_v19_reg_v22_retest_v1,ablation,dashboard_replay,-8.096810,-11.165928,3.069118,26.732402,0.000000,0
9,june_2026_common,smoke,canonical,dashboard_replay,-8.220079,-11.165928,2.945849,27.770055,0.000000,0


In [7]:
market_metrics_df = pd.DataFrame(
    [market_window_metrics(panel, label, start, end) for label, (start, end) in market_windows.items()]
)
market_metrics_df

,window,requested_start,start,end,rows,soxl_buyhold_return_pct,soxl_buyhold_max_drawdown_pct
0,ytd_2026_market_latest,2026-01-01,2026-01-02,2026-06-10,110,282.408962,43.473423
1,june_2026_market_latest,2026-06-01,2026-06-01,2026-06-10,8,-20.429018,35.606331


In [8]:
from contextlib import redirect_stdout
import io

from soxl_backtest import SOXLBacktest


def legacy_soxl_backtest_window(label: str, start: str, end: str) -> tuple[dict, pd.DataFrame, pd.DataFrame, str]:
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        backtest = SOXLBacktest(
            {
                "USE_LONGBRIDGE": False,
                "START_DATE": start,
                "END_DATE": end,
            }
        )
        backtest.run()

    base = float(backtest.config["INITIAL_CAPITAL"])
    daily = pd.DataFrame(
        {
            "window": label,
            "datetime": pd.to_datetime(backtest.df["date"]).dt.date.astype(str),
            "price": backtest.price_history,
            "strategy_nav": backtest.capital_history,
            "buyhold_nav": base * backtest.df["price"].astype(float).to_numpy() / float(backtest.df["price"].iloc[0]),
            "position": backtest.position_history,
            "panic_score": backtest.panic_score_history,
            "hv_20": backtest.hv_20_history,
            "hv_30": backtest.hv_30_history,
            "vol_coef": backtest.vol_coef_history,
        }
    )
    daily["strategy_nav_rebased"] = daily["strategy_nav"] / daily["strategy_nav"].iloc[0] * 100.0
    daily["buyhold_nav_rebased"] = daily["buyhold_nav"] / daily["buyhold_nav"].iloc[0] * 100.0
    daily["strategy_return_pct_from_window_start"] = daily["strategy_nav_rebased"] - 100.0
    daily["soxl_return_pct_from_window_start"] = daily["buyhold_nav_rebased"] - 100.0
    daily["excess_return_pct_from_window_start"] = (
        daily["strategy_return_pct_from_window_start"] - daily["soxl_return_pct_from_window_start"]
    )

    trades = pd.DataFrame(backtest.trades)
    if trades.empty:
        trades = pd.DataFrame(columns=["window", "date", "type", "price", "position", "shares", "total_value"])
    else:
        trades.insert(0, "window", label)

    metrics = {
        "window": label,
        "requested_start": start,
        "start": daily["datetime"].iloc[0],
        "end": daily["datetime"].iloc[-1],
        "rows": int(len(daily)),
        "initial_capital": base,
        "final_capital": float(daily["strategy_nav"].iloc[-1]),
        "soxl_buyhold_return_pct": float(daily["buyhold_nav"].iloc[-1] / daily["buyhold_nav"].iloc[0] * 100.0 - 100.0),
        "rulebase_return_pct": float(daily["strategy_nav"].iloc[-1] / base * 100.0 - 100.0),
        "excess_return_pct": float(
            daily["strategy_nav"].iloc[-1] / base * 100.0
            - daily["buyhold_nav"].iloc[-1] / daily["buyhold_nav"].iloc[0] * 100.0
        ),
        "rulebase_max_drawdown_pct": max_drawdown_pct(daily["strategy_nav"]),
        "soxl_buyhold_max_drawdown_pct": max_drawdown_pct(daily["buyhold_nav"]),
        "average_position_pct": float(daily["position"].astype(float).mean() * 100.0),
        "end_position_pct": float(daily["position"].astype(float).iloc[-1] * 100.0),
        "trade_count": int(len(backtest.trades)),
    }
    return metrics, daily, trades, buffer.getvalue()

In [9]:
legacy_outputs = []
legacy_logs = []
for label, start in [("legacy_ytd_2026_fresh_start", "2026-01-01"), ("legacy_june_2026_fresh_start", "2026-06-01")]:
    metrics, daily, trades, log = legacy_soxl_backtest_window(label, start, safe_backtest_end.isoformat())
    legacy_outputs.append((metrics, daily, trades))
    legacy_logs.append(f"## {label}\n\n```text\n{log.strip()}\n```\n")

legacy_metrics_df = pd.DataFrame([item[0] for item in legacy_outputs])
legacy_daily_df = pd.concat([item[1] for item in legacy_outputs], ignore_index=True)
legacy_trades_df = pd.concat([item[2] for item in legacy_outputs], ignore_index=True)


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(val) for key, val in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return str(value)
    return value


if not legacy_trades_df.empty and "trigger_conditions" in legacy_trades_df.columns:
    legacy_trades_df["trigger_conditions"] = legacy_trades_df["trigger_conditions"].apply(
        lambda value: json.dumps(json_safe(value), ensure_ascii=False) if isinstance(value, list) else ""
    )

legacy_metrics_path = OUTPUT_DIR / "legacy_soxl_backtest_window_metrics.csv"
legacy_daily_path = OUTPUT_DIR / "legacy_soxl_backtest_window_daily.csv"
legacy_trades_path = OUTPUT_DIR / "legacy_soxl_backtest_trades.csv"
legacy_log_path = OUTPUT_DIR / "legacy_soxl_backtest_run_log.md"

legacy_metrics_df.to_csv(legacy_metrics_path, index=False)
legacy_daily_df.to_csv(legacy_daily_path, index=False)
legacy_trades_df.to_csv(legacy_trades_path, index=False)
legacy_log_path.write_text("\n".join(legacy_logs) + "\n")

print(legacy_metrics_path.relative_to(ROOT))
print(legacy_daily_path.relative_to(ROOT))
print(legacy_trades_path.relative_to(ROOT))
print(legacy_log_path.relative_to(ROOT))
legacy_metrics_df

runtime/production/card/soxl_backtest_jan_june_windows_v1/legacy_soxl_backtest_window_metrics.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/legacy_soxl_backtest_window_daily.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/legacy_soxl_backtest_trades.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/legacy_soxl_backtest_run_log.md


,window,requested_start,start,end,rows,initial_capital,final_capital,soxl_buyhold_return_pct,rulebase_return_pct,excess_return_pct,rulebase_max_drawdown_pct,soxl_buyhold_max_drawdown_pct,average_position_pct,end_position_pct,trade_count
0,legacy_ytd_2026_fresh_start,2026-01-01,2026-01-02,2026-06-09,109,100000.0,232077.977951,326.926318,132.077978,-194.848340,13.554089,43.473423,16.220734,15.91,3
1,legacy_june_2026_fresh_start,2026-06-01,2026-06-01,2026-06-09,7,100000.0,103138.960000,-11.165925,3.138960,14.304885,1.528209,34.932630,12.831429,29.94,1


In [10]:
window_definitions_path = OUTPUT_DIR / "window_definitions.csv"
current_metrics_path = OUTPUT_DIR / "current_model_window_metrics.csv"
all_metrics_path = OUTPUT_DIR / "all_model_window_metrics.csv"
market_metrics_path = OUTPUT_DIR / "market_only_latest_metrics.csv"
current_daily_path = OUTPUT_DIR / "current_model_window_daily.csv"
ytd_daily_path = OUTPUT_DIR / "current_model_ytd_2026_common_daily.csv"
june_daily_path = OUTPUT_DIR / "current_model_june_2026_common_daily.csv"

window_definitions.to_csv(window_definitions_path, index=False)
current_metrics_df.to_csv(current_metrics_path, index=False)
all_model_metrics_df.to_csv(all_metrics_path, index=False)
market_metrics_df.to_csv(market_metrics_path, index=False)
current_window_daily.to_csv(current_daily_path, index=False)
current_window_daily[current_window_daily["window"] == "ytd_2026_common"].to_csv(ytd_daily_path, index=False)
current_window_daily[current_window_daily["window"] == "june_2026_common"].to_csv(june_daily_path, index=False)

for path in [
    window_definitions_path,
    current_metrics_path,
    all_metrics_path,
    market_metrics_path,
    current_daily_path,
    ytd_daily_path,
    june_daily_path,
]:
    print(path.relative_to(ROOT))

runtime/production/card/soxl_backtest_jan_june_windows_v1/window_definitions.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/current_model_window_metrics.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/all_model_window_metrics.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/market_only_latest_metrics.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/current_model_window_daily.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/current_model_ytd_2026_common_daily.csv
runtime/production/card/soxl_backtest_jan_june_windows_v1/current_model_june_2026_common_daily.csv


In [11]:
ytd = current_metrics_df.loc[current_metrics_df["window"] == "ytd_2026_common"].iloc[0]
june = current_metrics_df.loc[current_metrics_df["window"] == "june_2026_common"].iloc[0]
market_ytd = market_metrics_df.loc[market_metrics_df["window"] == "ytd_2026_market_latest"].iloc[0]
market_june = market_metrics_df.loc[market_metrics_df["window"] == "june_2026_market_latest"].iloc[0]

ytd_ranked = all_model_metrics_df[all_model_metrics_df["window"] == "ytd_2026_common"].sort_values(
    "model_return_pct", ascending=False
)
june_ranked = all_model_metrics_df[all_model_metrics_df["window"] == "june_2026_common"].sort_values(
    "model_return_pct", ascending=False
)

top_ytd = ytd_ranked.iloc[0]
top_june = june_ranked.iloc[0]
laggard_ytd = ytd_ranked.iloc[-1]
laggard_june = june_ranked.iloc[-1]

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "experiment": EXPERIMENT,
    "direction": "production",
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "source_current_model_daily": str(CURRENT_DAILY_PATH.relative_to(ROOT)),
    "source_all_model_daily_nav": str(ALL_MODEL_DAILY_PATH.relative_to(ROOT)),
    "source_panel": str(PANEL_PATH.relative_to(ROOT)),
    "safe_backtest_end": safe_backtest_end.isoformat(),
    "latest_data_date": latest_data_date.isoformat(),
    "current_model_metrics": current_metrics_df.to_dict(orient="records"),
    "all_model_rows": int(len(all_model_metrics_df)),
    "all_model_ytd_leader": top_ytd[rank_cols].to_dict(),
    "all_model_june_leader": top_june[rank_cols].to_dict(),
    "all_model_ytd_laggard": laggard_ytd[rank_cols].to_dict(),
    "all_model_june_laggard": laggard_june[rank_cols].to_dict(),
    "market_only_latest_metrics": market_metrics_df.to_dict(orient="records"),
    "legacy_soxl_backtest_metrics": legacy_metrics_df.to_dict(orient="records"),
    "artifacts": {
        "notebook": str((OUTPUT_DIR / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "window_definitions": str(window_definitions_path.relative_to(ROOT)),
        "current_model_window_metrics": str(current_metrics_path.relative_to(ROOT)),
        "all_model_window_metrics": str(all_metrics_path.relative_to(ROOT)),
        "market_only_latest_metrics": str(market_metrics_path.relative_to(ROOT)),
        "legacy_soxl_backtest_window_metrics": str(legacy_metrics_path.relative_to(ROOT)),
        "legacy_soxl_backtest_window_daily": str(legacy_daily_path.relative_to(ROOT)),
        "legacy_soxl_backtest_trades": str(legacy_trades_path.relative_to(ROOT)),
        "legacy_soxl_backtest_run_log": str(legacy_log_path.relative_to(ROOT)),
        "current_model_window_daily": str(current_daily_path.relative_to(ROOT)),
        "current_model_ytd_2026_common_daily": str(ytd_daily_path.relative_to(ROOT)),
        "current_model_june_2026_common_daily": str(june_daily_path.relative_to(ROOT)),
        "summary": str((OUTPUT_DIR / "summary.json").relative_to(ROOT)),
        "conclusion": str((OUTPUT_DIR / "conclusion.md").relative_to(ROOT)),
    },
}

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")
print(summary_path.relative_to(ROOT))

runtime/production/card/soxl_backtest_jan_june_windows_v1/summary.json


In [12]:
def fmt_pct(value: float) -> str:
    return f"{float(value):.2f}%"

legacy_ytd = legacy_metrics_df.loc[legacy_metrics_df["window"] == "legacy_ytd_2026_fresh_start"].iloc[0]
legacy_june = legacy_metrics_df.loc[legacy_metrics_df["window"] == "legacy_june_2026_fresh_start"].iloc[0]

conclusion_md = f"""## Conclusion

- Purpose: this notebook packages the SOXL backtest data for the two requested 2026 windows, using rebased window-level NAV so the January and June slices are directly usable.
- Model comparison data source: `{CURRENT_DAILY_PATH.relative_to(ROOT)}` for the current production composer, plus `{ALL_MODEL_DAILY_PATH.relative_to(ROOT)}` for dashboard replay peers.
- Date rule: model comparisons stop at `{safe_backtest_end.isoformat()}` because dashboard replays are safe only through that date; market-only SOXL data is fresh through `{latest_data_date.isoformat()}`.
- January/YTD common window: requested `2026-01-01`, actual first trading row `{ytd['start']}`, end `{ytd['end']}`, rows `{int(ytd['rows'])}`. SOXL buy-and-hold returned `{fmt_pct(ytd['soxl_buyhold_return_pct'])}`; current production model returned `{fmt_pct(ytd['model_return_pct'])}`; excess `{float(ytd['excess_return_pct']):.2f}pp`. Model max drawdown `{fmt_pct(ytd['model_max_drawdown_pct'])}` vs SOXL `{fmt_pct(ytd['soxl_buyhold_max_drawdown_pct'])}`.
- June common window: requested `2026-06-01`, actual first trading row `{june['start']}`, end `{june['end']}`, rows `{int(june['rows'])}`. SOXL buy-and-hold returned `{fmt_pct(june['soxl_buyhold_return_pct'])}`; current production model returned `{fmt_pct(june['model_return_pct'])}`; excess `{float(june['excess_return_pct']):.2f}pp`. Model max drawdown `{fmt_pct(june['model_max_drawdown_pct'])}` vs SOXL `{fmt_pct(june['soxl_buyhold_max_drawdown_pct'])}`.
- Legacy `soxl_backtest` YTD fresh-start run: `{legacy_ytd['start']}` to `{legacy_ytd['end']}`, rulebase returned `{fmt_pct(legacy_ytd['rulebase_return_pct'])}` vs SOXL buy-and-hold `{fmt_pct(legacy_ytd['soxl_buyhold_return_pct'])}`, excess `{float(legacy_ytd['excess_return_pct']):.2f}pp`, trades `{int(legacy_ytd['trade_count'])}`, end position `{fmt_pct(legacy_ytd['end_position_pct'])}`.
- Legacy `soxl_backtest` June fresh-start run: `{legacy_june['start']}` to `{legacy_june['end']}`, rulebase returned `{fmt_pct(legacy_june['rulebase_return_pct'])}` vs SOXL buy-and-hold `{fmt_pct(legacy_june['soxl_buyhold_return_pct'])}`, excess `{float(legacy_june['excess_return_pct']):.2f}pp`, trades `{int(legacy_june['trade_count'])}`, end position `{fmt_pct(legacy_june['end_position_pct'])}`. This June run starts a fresh rulebase account on June 1, so use it as a standalone scenario rather than a continuation of the January run.
- All-model YTD leader: `{top_ytd['profile']}` returned `{fmt_pct(top_ytd['model_return_pct'])}` with max drawdown `{fmt_pct(top_ytd['model_max_drawdown_pct'])}`. YTD laggard: `{laggard_ytd['profile']}` returned `{fmt_pct(laggard_ytd['model_return_pct'])}`.
- All-model June leader: `{top_june['profile']}` returned `{fmt_pct(top_june['model_return_pct'])}` with max drawdown `{fmt_pct(top_june['model_max_drawdown_pct'])}`. June laggard: `{laggard_june['profile']}` returned `{fmt_pct(laggard_june['model_return_pct'])}`.
- Market-only latest through `{latest_data_date.isoformat()}`: SOXL buy-and-hold returned `{fmt_pct(market_ytd['soxl_buyhold_return_pct'])}` YTD and `{fmt_pct(market_june['soxl_buyhold_return_pct'])}` for June. Do not compare these directly to model rows because the model replay common end is `{safe_backtest_end.isoformat()}`.
- Primary reusable outputs: `current_model_ytd_2026_common_daily.csv`, `current_model_june_2026_common_daily.csv`, `current_model_window_metrics.csv`, `all_model_window_metrics.csv`, `legacy_soxl_backtest_window_metrics.csv`, `legacy_soxl_backtest_window_daily.csv`, and `legacy_soxl_backtest_trades.csv` in this card directory.
"""

conclusion_path = OUTPUT_DIR / "conclusion.md"
conclusion_path.write_text(conclusion_md + "\n")
display(Markdown(conclusion_md))

## Conclusion

- Purpose: this notebook packages the SOXL backtest data for the two requested 2026 windows, using rebased window-level NAV so the January and June slices are directly usable.
- Model comparison data source: `runtime/production/card/current_model_ytd_june_comparison_v1/current_model_daily.csv` for the current production composer, plus `runtime/production/card/current_model_ytd_june_comparison_v1/all_model_daily_nav.csv` for dashboard replay peers.
- Date rule: model comparisons stop at `2026-06-09` because dashboard replays are safe only through that date; market-only SOXL data is fresh through `2026-06-10`.
- January/YTD common window: requested `2026-01-01`, actual first trading row `2026-01-02`, end `2026-06-09`, rows `109`. SOXL buy-and-hold returned `326.93%`; current production model returned `3.25%`; excess `-323.68pp`. Model max drawdown `21.09%` vs SOXL `43.47%`.
- June common window: requested `2026-06-01`, actual first trading row `2026-06-01`, end `2026-06-09`, rows `7`. SOXL buy-and-hold returned `-11.17%`; current production model returned `0.00%`; excess `11.17pp`. Model max drawdown `0.00%` vs SOXL `34.93%`.
- Legacy `soxl_backtest` YTD fresh-start run: `2026-01-02` to `2026-06-09`, rulebase returned `132.08%` vs SOXL buy-and-hold `326.93%`, excess `-194.85pp`, trades `3`, end position `15.91%`.
- Legacy `soxl_backtest` June fresh-start run: `2026-06-01` to `2026-06-09`, rulebase returned `3.14%` vs SOXL buy-and-hold `-11.17%`, excess `14.30pp`, trades `1`, end position `29.94%`. This June run starts a fresh rulebase account on June 1, so use it as a standalone scenario rather than a continuation of the January run.
- All-model YTD leader: `smoke` returned `224.36%` with max drawdown `30.35%`. YTD laggard: `v19reg_v22_budget_usage_bull_guard_v1` returned `3.25%`.
- All-model June leader: `v19reg_v22_budget_usage_bull_guard_v1` returned `0.00%` with max drawdown `0.00%`. June laggard: `state_conditioned_drawdown_rebound_reward_train_v1` returned `-9.80%`.
- Market-only latest through `2026-06-10`: SOXL buy-and-hold returned `282.41%` YTD and `-20.43%` for June. Do not compare these directly to model rows because the model replay common end is `2026-06-09`.
- Primary reusable outputs: `current_model_ytd_2026_common_daily.csv`, `current_model_june_2026_common_daily.csv`, `current_model_window_metrics.csv`, `all_model_window_metrics.csv`, `legacy_soxl_backtest_window_metrics.csv`, `legacy_soxl_backtest_window_daily.csv`, and `legacy_soxl_backtest_trades.csv` in this card directory.
